# ChestX-ray14 classification: techniques from the papers

This notebook builds a **14-label** chest-radiograph classifier using NIH ChestX-ray14. One image can have several findings at once, so the model predicts each finding independently rather than choosing one mutually exclusive class.

It is an educational implementation based on several papers, not a reproduction of any paper's reported result. The NIH labels were mined from radiology reports and can be noisy; predictions and attention maps are not a diagnosis or validated localization.

## What I read, and what this notebook uses

| Paper | Technique or lesson | How it informs this notebook |
|---|---|---|
| [Wang et al., CVPR 2017 — ChestX-ray8](https://openaccess.thecvf.com/content_cvpr_2017/html/Wang_ChestX-ray8_Hospital-Scale_Chest_CVPR_2017_paper.html) | Introduced the large NIH image collection and weakly supervised labels/localization. Labels come from reports, not pixel-level expert annotations. | Treat labels as image-level targets; preserve patient separation; avoid claiming attention maps are proven lesion locations. |
| [Yao et al., 2017 — Learning to diagnose from scratch by exploiting dependencies among labels](https://arxiv.org/abs/1710.10501) | Uses an LSTM to model relationships among the 14 findings. | Motivates the multi-label setup. This notebook does not implement label-sequence modeling; each finding is predicted independently. |
| [Rajpurkar et al., 2017 — CheXNet](https://arxiv.org/abs/1711.05225) | ImageNet-pretrained DenseNet-121; 224×224 images; sigmoid outputs and binary cross-entropy for the 14-label extension; reports per-finding AUROC. | Backbone, image scale, multi-label loss, and per-label evaluation. |
| [Ge et al., 2018 — Chest X-rays Classification: A Multi-Label and Fine-Grained Problem](https://arxiv.org/abs/1807.07247) | Studies a custom multi-label softmax loss and fine-grained/bilinear features to address co-occurring labels and imbalance. | Highlights that ordinary BCE is a baseline, not the only sensible loss. This notebook keeps BCE transparent and offers optional capped positive weights. |
| [Guan & Huang, 2020 — Category-wise Residual Attention Learning](https://doi.org/10.1016/j.patrec.2018.10.027) | Learns category-specific attention so different diseases can use different image evidence. | The head creates one spatial attention map per finding. It is a simplified attention head, not CRAL's residual block. |
| [Wang et al., 2021 — Triple Attention Learning (A³Net)](https://doi.org/10.1016/j.media.2020.101846) | DenseNet-121 with channel-wise, element-wise, and scale-wise attention; uses ChestX-ray14's patient-level official split. | The notebook uses ImageNet-pretrained DenseNet features, a channel gate, category-specific spatial attention, and the official held-out patient split. It does not reproduce A³Net's scale-wise module or ensemble. |

### Notebook workflow

1. Attach the Kaggle dataset, download it with KaggleHub, or select an extracted local copy.
2. Audit labels and image files; use the supplied official train/validation and test image lists.
3. Create a patient-disjoint validation set from the official training patients.
4. Fine-tune DenseNet-121 with category-specific attention and multi-label BCE.
5. Select the checkpoint using validation AUROC, then evaluate once on the untouched test patients.
6. Inspect per-label AUROC / average precision and an attention overlay; save checkpoints and tables to the run folder.

The original archive is large. This notebook expects the images and metadata to be downloaded and extracted beforehand; it does not silently download tens of gigabytes or require Kaggle credentials.

## 1. Install the small dependencies

PyTorch and torchvision are required. Google Colab usually provides them; for a local environment, install a PyTorch build appropriate for your hardware from the [official PyTorch installation page](https://pytorch.org/get-started/locally/).

In [ ]:
%pip install -q pandas scikit-learn matplotlib tqdm pillow kagglehub

In [ ]:
from pathlib import Path
import os, random, json, math, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.models import densenet121, DenseNet121_Weights
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

print('PyTorch:', torch.__version__)
print('Torchvision import succeeded')
print('CUDA available:', torch.cuda.is_available())


## 2. Import ChestX-ray14 from Kaggle

Dataset page: [NIH Chest X-rays on Kaggle](https://www.kaggle.com/datasets/nih-chest-xrays/data). This notebook needs the **full dataset**, including `Data_Entry_2017*.csv`, `train_val_list.txt`, `test_list.txt`, and the PNG images.

Choose one route in the next cell:

- **Kaggle Notebook:** click **Add Input**, attach “NIH Chest X-rays”, and use `DATA_SOURCE="kaggle_input"`. KaggleHub is already authenticated inside a Kaggle Notebook, and attached files are available under `/kaggle/input`.
- **Google Colab or local Jupyter:** keep `DATA_SOURCE="kagglehub"`. KaggleHub downloads the pinned version of the dataset and returns its extracted directory. The full data is large; allow at least 110 GiB free for download, extraction, and cached files.
- **Already downloaded:** set `DATA_SOURCE="local"` and point `LOCAL_DATA_ROOT` at the extracted folder.

If Kaggle requests consent or authentication, sign in to Kaggle and accept the dataset conditions. In Colab, store `KAGGLE_API_TOKEN` in Colab Secrets and read it from the secret store, or use `kagglehub.login()`. Never paste a token into a notebook cell or commit it to a repository. See [KaggleHub authentication and download docs](https://github.com/Kaggle/kagglehub#authenticate) and [Kaggle API docs](https://www.kaggle.com/docs/api).

In [ ]:
# Choose: 'kagglehub', 'kaggle_input', or 'local'.
DATA_SOURCE = 'kagglehub'
KAGGLE_HANDLE = 'nih-chest-xrays/data/versions/3'
LOCAL_DATA_ROOT = Path('/content/nih-chestxray14')  # edit for your extracted local copy

if DATA_SOURCE == 'kagglehub':
    import kagglehub
    DATA_ROOT = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
elif DATA_SOURCE == 'kaggle_input':
    DATA_ROOT = Path('/kaggle/input')
elif DATA_SOURCE == 'local':
    DATA_ROOT = LOCAL_DATA_ROOT
else:
    raise ValueError("DATA_SOURCE must be 'kagglehub', 'kaggle_input', or 'local'")

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset location does not exist: {DATA_ROOT}')
metadata_candidates = sorted(set(DATA_ROOT.rglob('Data_Entry_2017*.csv')))
if len(metadata_candidates) != 1:
    raise FileNotFoundError(
        f'Expected exactly one Data_Entry_2017*.csv under {DATA_ROOT}; '
        f'found {len(metadata_candidates)}. If using Kaggle inputs, set DATA_ROOT to the NIH dataset folder.'
    )
for required_file in ['train_val_list.txt', 'test_list.txt']:
    matches = list(DATA_ROOT.rglob(required_file))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {required_file} under {DATA_ROOT}; found {len(matches)}')
if next(DATA_ROOT.rglob('*.png'), None) is None:
    raise FileNotFoundError(f'No PNG images found under {DATA_ROOT}. Extract all image archives first.')

print('Dataset source:', DATA_SOURCE)
print('Resolved dataset directory:', DATA_ROOT.resolve())
print('Metadata:', metadata_candidates[0])
print('Official split lists found; PNG images found.')


## 3. Configure training and choose where results are saved

The default saves results to Google Drive when running in Colab, `/kaggle/working` in a Kaggle Notebook, or a local output folder elsewhere. A run folder stores the training configuration, best and latest checkpoints, epoch history, split summary, per-label test metrics, and predictions.

Use a new `RUN_NAME` for a new experiment or changed settings. To continue an interrupted run, keep the same run name and settings, set `RESUME_TRAINING=True`, then rerun the notebook from the top. The last checkpoint is saved at the end of each epoch. Increasing `EPOCHS` is allowed when resuming; the other training settings must match.

In Colab, saving to Drive keeps outputs after a runtime reset. The notebook itself is a separate file: use **File → Save a copy in Drive** (or Ctrl/Cmd+S) to save code and displayed cell outputs.

In [ ]:
# Set False to save under the current runtime/work directory instead of Google Drive.
SAVE_TO_GOOGLE_DRIVE = True
RUN_NAME = 'chestxray14_densenet_attention_seed42_v1'
RESUME_TRAINING = False

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0  # portable default; increase to 2-4 if your runtime benefits
EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
VALIDATION_FRACTION = 0.10
USE_POS_WEIGHT = False
MAX_POS_WEIGHT = 10.0

LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural_Thickening', 'Hernia'
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if SAVE_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        OUTPUT_BASE = Path('/content/drive/MyDrive/ChestXray14Runs')
    except ImportError:
        print('Google Colab not detected; saving to the runtime/work directory instead.')
        OUTPUT_BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
else:
    OUTPUT_BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()

OUTPUT_DIR = OUTPUT_BASE / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_CONFIG = {
    'seed': SEED, 'image_size': IMAGE_SIZE, 'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS, 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'validation_fraction': VALIDATION_FRACTION,
    'use_pos_weight': USE_POS_WEIGHT, 'max_pos_weight': MAX_POS_WEIGHT,
    'labels': LABELS, 'dataset_source': DATA_SOURCE, 'dataset_handle': KAGGLE_HANDLE,
    'dataset_root': str(DATA_ROOT.resolve()),
}
run_config_path = OUTPUT_DIR / 'run_config.json'
if run_config_path.exists():
    saved_config = json.loads(run_config_path.read_text(encoding='utf-8'))
    if not RESUME_TRAINING:
        raise FileExistsError(
            f'{OUTPUT_DIR} already has a run. Set RESUME_TRAINING=True to resume with the same settings, '
            'or choose a new RUN_NAME for a fresh run.'
        )
    changed = [key for key, value in RUN_CONFIG.items()
               if key != 'epochs' and saved_config.get(key) != value]
    if changed:
        raise ValueError(f'Cannot resume because these settings changed: {changed}. Use a new RUN_NAME.')
    if EPOCHS < int(saved_config.get('epochs', EPOCHS)):
        raise ValueError('When resuming, EPOCHS must be at least the prior configured epoch count.')
elif RESUME_TRAINING:
    raise FileNotFoundError(f'Cannot resume: no run_config.json found in {OUTPUT_DIR}')

run_config_path.write_text(json.dumps(RUN_CONFIG, indent=2), encoding='utf-8')
print('Device:', DEVICE)
print('Run folder:', OUTPUT_DIR.resolve())
print('Resume:', RESUME_TRAINING)


## 4. Read metadata and audit the image collection

The `No Finding` string is not one of the 14 disease outputs: it becomes an all-zero disease vector. That means “no finding was reported in this dataset label,” not proof that the image is clinically normal. Each disease is encoded as 0/1 and multiple positives are allowed.

In [ ]:
csv_candidates = metadata_candidates  # found once in the Kaggle/local import cell above
if len(csv_candidates) != 1:
    raise FileNotFoundError(
        f'Expected exactly one Data_Entry_2017*.csv under {DATA_ROOT}; found {len(csv_candidates)}. '
        'Point DATA_ROOT at one full dataset copy.'
    )
metadata_path = csv_candidates[0]
metadata = pd.read_csv(metadata_path, dtype={'Patient ID': str})
required = {'Image Index', 'Finding Labels', 'Patient ID'}
missing_columns = required - set(metadata.columns)
if missing_columns:
    raise ValueError(f'Metadata is missing required columns: {sorted(missing_columns)}')
if metadata['Image Index'].duplicated().any():
    raise ValueError('Image Index contains duplicate rows')

unknown_labels = set()
for value in metadata['Finding Labels'].dropna().astype(str):
    unknown_labels.update(set(value.split('|')) - set(LABELS) - {'No Finding'})
if unknown_labels:
    raise ValueError(f'Unexpected labels: {sorted(unknown_labels)}')
if metadata[['Image Index', 'Finding Labels', 'Patient ID']].isna().any().any():
    raise ValueError('Required metadata fields contain missing values')

for label in LABELS:
    metadata[label] = metadata['Finding Labels'].map(
        lambda value: int(label in str(value).split('|'))
    ).astype('uint8')
metadata['Patient ID'] = metadata['Patient ID'].astype(str).str.strip()
if (metadata['Patient ID'] == '').any():
    raise ValueError('Patient ID contains an empty value')

# Search the NIH image folders beside the metadata instead of recursively indexing
# every PNG under DATA_ROOT (which may be the entire /kaggle/input mount).
expected_images = set(metadata['Image Index'])
dataset_dir = metadata_path.parent
image_dirs = set()
for candidate in dataset_dir.glob('images*'):
    if candidate.is_dir():
        nested_images = candidate / 'images'
        image_dirs.add(nested_images if nested_images.is_dir() else candidate)
for candidate in dataset_dir.glob('images'):
    if candidate.is_dir():
        image_dirs.add(candidate)

# Some mirrors add one wrapper folder around the standard images_###/images layout.
if not image_dirs:
    image_dirs = {candidate for candidate in dataset_dir.rglob('images') if candidate.is_dir()}
if not image_dirs:
    raise FileNotFoundError(
        f'No images* folders found beside the metadata file at {dataset_dir}. '
        'Check that all NIH image archives were extracted.'
    )

image_paths = {}
for image_dir in sorted(image_dirs):
    print(f'Indexing {image_dir} ...')
    with os.scandir(image_dir) as entries:
        for entry in tqdm(entries, desc=image_dir.name, unit='files', leave=False):
            if entry.is_file() and entry.name in expected_images:
                if entry.name in image_paths:
                    raise ValueError(f'Duplicate image basename found: {entry.name}')
                image_paths[entry.name] = str(Path(entry.path).resolve())
metadata['path'] = metadata['Image Index'].map(image_paths)
missing_images = int(metadata['path'].isna().sum())
print(f'Metadata file: {metadata_path}')
print(f'Images in metadata: {len(metadata):,}')
print(f'Unique patients: {metadata["Patient ID"].nunique():,}')
print(f'Extracted PNG files found: {len(image_paths):,}')
print(f'Metadata rows with missing image files: {missing_images:,}')
metadata_sha256 = hashlib.sha256(metadata_path.read_bytes()).hexdigest()
dataset_info = {'metadata_file': str(metadata_path), 'metadata_sha256': metadata_sha256,
                'images': len(metadata), 'patients': int(metadata['Patient ID'].nunique()),
                'dataset_source': DATA_SOURCE, 'dataset_handle': KAGGLE_HANDLE}
dataset_info_path = OUTPUT_DIR / 'dataset_info.json'
if dataset_info_path.exists():
    previous_info = json.loads(dataset_info_path.read_text(encoding='utf-8'))
    if RESUME_TRAINING and previous_info.get('metadata_sha256') != metadata_sha256:
        raise ValueError('Metadata changed since the previous run; use a new RUN_NAME.')
dataset_info_path.write_text(json.dumps(dataset_info, indent=2), encoding='utf-8')
if missing_images:
    raise FileNotFoundError('Extract all image archives before running this full-dataset notebook.')


## 5. Use official patient-level lists and make validation patients

The test images come directly from `test_list.txt`; they stay out of training and checkpoint selection. Validation patients are sampled only from `train_val_list.txt`, stratifying patients into all-zero versus at-least-one-finding groups. This preserves patient separation, although it cannot perfectly stratify all 14 labels at once.

In [ ]:
def find_one(filename):
    matches = list(DATA_ROOT.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(f'Expected exactly one {filename} under {DATA_ROOT}; found {len(matches)}')
    return matches[0]

def read_image_list(filename):
    values = [line.strip() for line in find_one(filename).read_text().splitlines() if line.strip()]
    if len(values) != len(set(values)):
        raise ValueError(f'{filename} contains duplicate image names')
    return set(values)

trainval_ids = read_image_list('train_val_list.txt')
test_ids = read_image_list('test_list.txt')
if trainval_ids & test_ids:
    raise ValueError('Official train/validation and test image lists overlap')
metadata_ids = set(metadata['Image Index'])
if metadata_ids != trainval_ids | test_ids:
    raise ValueError(
        f'Official lists do not exactly cover metadata: {len(metadata_ids - (trainval_ids | test_ids))} '
        f'metadata-only, {len((trainval_ids | test_ids) - metadata_ids)} list-only image IDs.'
    )

trainval_df = metadata[metadata['Image Index'].isin(trainval_ids)].copy()
test_df = metadata[metadata['Image Index'].isin(test_ids)].copy()
patient_table = trainval_df.groupby('Patient ID')[LABELS].max()
patient_any_finding = (patient_table.sum(axis=1) > 0).astype(int)
train_patients, val_patients = train_test_split(
    patient_table.index.to_numpy(),
    test_size=VALIDATION_FRACTION,
    random_state=SEED,
    stratify=patient_any_finding.to_numpy()
)
train_df = trainval_df[trainval_df['Patient ID'].isin(train_patients)].copy()
val_df = trainval_df[trainval_df['Patient ID'].isin(val_patients)].copy()

for left_name, left, right_name, right in [
    ('train', train_df, 'validation', val_df),
    ('train', train_df, 'test', test_df),
    ('validation', val_df, 'test', test_df)
]:
    overlap = set(left['Patient ID']) & set(right['Patient ID'])
    if overlap:
        raise ValueError(f'Patient leakage between {left_name} and {right_name}: {len(overlap)} patients')

split_summary = pd.DataFrame([
    {'split': name, 'images': len(frame), 'patients': frame['Patient ID'].nunique(),
     'images_with_any_positive_label': int(frame[LABELS].any(axis=1).sum())}
    for name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]
]).set_index('split')
display(split_summary)
split_summary.to_csv(OUTPUT_DIR / 'split_summary.csv')


In [ ]:
label_audit = pd.DataFrame({
    'positive_images': train_df[LABELS].sum().astype(int),
    'train_prevalence': train_df[LABELS].mean(),
    'validation_positive_images': val_df[LABELS].sum().astype(int),
    'test_positive_images': test_df[LABELS].sum().astype(int),
}).sort_values('positive_images')
display(label_audit)
label_audit.to_csv(OUTPUT_DIR / 'label_support.csv')
label_audit['positive_images'].plot(kind='barh', figsize=(8, 6), title='Positive training images by finding')
plt.xlabel('Number of images with the label')
plt.tight_layout()
plt.show()


## 6. Preprocess images and create data loaders

The model uses ImageNet normalization to match its pretrained backbone. Images are resized without cropping, with padding if needed. Training uses small rotations and translations. Horizontal flipping is left out so that laterality is preserved.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def fit_to_square(image, size):
    image = ImageOps.contain(image, (size, size))
    canvas = Image.new('RGB', (size, size), color=(0, 0, 0))
    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    return canvas

class ChestXray14Dataset(Dataset):
    def __init__(self, frame, training=False):
        self.frame = frame.reset_index(drop=True)
        self.training = training
        ops = [T.Lambda(lambda image: fit_to_square(image, IMAGE_SIZE))]
        if training:
            ops.append(T.RandomAffine(degrees=5, translate=(0.02, 0.02), fill=(0, 0, 0)))
        ops.extend([T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
        self.transform = T.Compose(ops)
        self.targets = self.frame[LABELS].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        path = self.frame.iloc[index]['path']
        with Image.open(path) as image:
            image = image.convert('RGB')
            x = self.transform(image)
        y = torch.from_numpy(self.targets[index])
        return x, y

loader_options = {
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'pin_memory': torch.cuda.is_available(),
}
train_loader = DataLoader(ChestXray14Dataset(train_df, training=True), shuffle=True, **loader_options)
val_loader = DataLoader(ChestXray14Dataset(val_df, training=False), shuffle=False, **loader_options)
test_loader = DataLoader(ChestXray14Dataset(test_df, training=False), shuffle=False, **loader_options)

example_x, example_y = next(iter(train_loader))
print('Image batch:', tuple(example_x.shape), '| label batch:', tuple(example_y.shape))
print('Number of positive labels in first batch:', int(example_y.sum().item()))


## 7. Build DenseNet-121 with channel and category-specific spatial attention

CheXNet supplies the ImageNet-pretrained DenseNet-121 backbone. The compact head borrows the attention idea from CRAL and A³Net:

- a squeeze-and-excitation style channel gate reweights feature channels;
- a separate spatial attention map is learned for each of the 14 findings;
- each map pools the feature grid into a class-specific representation before its logit is computed.

This is intentionally a small, understandable adaptation. It is not the exact CRAL residual attention block or A³Net's three-scale attention and multi-scale ensemble.

In [ ]:
class ChannelGate(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 32)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1),
            nn.Sigmoid()
        )

    def forward(self, features):
        return features * self.mlp(self.pool(features))

class CategoryAttentionHead(nn.Module):
    def __init__(self, channels, n_labels, dropout=0.2):
        super().__init__()
        self.channel_gate = ChannelGate(channels)
        self.attention = nn.Conv2d(channels, n_labels, kernel_size=1)
        self.dropout = nn.Dropout(dropout)
        self.class_weight = nn.Parameter(torch.empty(n_labels, channels))
        self.class_bias = nn.Parameter(torch.zeros(n_labels))
        nn.init.xavier_uniform_(self.class_weight)

    def forward(self, features):
        features = self.channel_gate(features)
        batch, channels, height, width = features.shape
        attention_logits = self.attention(features).reshape(batch, len(LABELS), height * width)
        attention = torch.softmax(attention_logits, dim=-1)
        flat_features = features.flatten(2).transpose(1, 2)  # [batch, locations, channels]
        pooled = torch.bmm(attention, flat_features)          # [batch, labels, channels]
        pooled = self.dropout(pooled)
        logits = (pooled * self.class_weight.unsqueeze(0)).sum(dim=-1) + self.class_bias
        return logits, attention.reshape(batch, len(LABELS), height, width)

class DenseNet121CategoryAttention(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = DenseNet121_Weights.DEFAULT if pretrained else None
        backbone = densenet121(weights=weights)
        self.features = backbone.features
        self.head = CategoryAttentionHead(channels=1024, n_labels=len(LABELS))

    def forward(self, x):
        features = self.features(x)
        features = F.relu(features, inplace=True)
        return self.head(features)

model = DenseNet121CategoryAttention(pretrained=True).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


## 8. Train and choose the checkpoint on validation data

The output is 14 logits; `BCEWithLogitsLoss` applies a separate binary loss to each disease and allows co-occurrence. The training loop tracks per-label AUROC and average precision. The best checkpoint is chosen by validation macro AUROC. The test set is not used here.

Training artifacts are written into the selected run folder. `last_checkpoint.pt` is updated after every completed epoch for resuming. `best_densenet121_category_attention.pt` stores the validation-selected model. The notebook also updates `epoch_history.csv` after every epoch.

In [ ]:
train_positive = train_df[LABELS].sum(axis=0).to_numpy(dtype=np.float32)
train_negative = len(train_df) - train_positive
positive_weights = np.clip(train_negative / np.maximum(train_positive, 1.0), 1.0, MAX_POS_WEIGHT)
positive_weights = torch.tensor(positive_weights, dtype=torch.float32, device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weights if USE_POS_WEIGHT else None)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.1, patience=1
)

@torch.no_grad()
def collect_predictions(loader):
    model.eval()
    targets_all, probabilities_all = [], []
    total_loss, total_examples = 0.0, 0
    for images, targets in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        logits, _ = model(images)
        loss = criterion(logits, targets)
        probabilities = torch.sigmoid(logits)
        total_loss += loss.item() * len(images)
        total_examples += len(images)
        targets_all.append(targets.cpu().numpy())
        probabilities_all.append(probabilities.cpu().numpy())
    y_true = np.concatenate(targets_all, axis=0)
    y_score = np.concatenate(probabilities_all, axis=0)
    return total_loss / max(total_examples, 1), y_true, y_score

def multilabel_metrics(y_true, y_score):
    aucs, aps = [], []
    rows = []
    for index, label in enumerate(LABELS):
        y = y_true[:, index]
        auc = roc_auc_score(y, y_score[:, index]) if np.unique(y).size == 2 else np.nan
        ap = average_precision_score(y, y_score[:, index]) if y.sum() > 0 else np.nan
        aucs.append(auc)
        aps.append(ap)
        rows.append({'label': label, 'AUROC': auc, 'Average precision': ap,
                     'positive examples': int(y.sum()), 'prevalence': float(y.mean())})
    return float(np.nanmean(aucs)), float(np.nanmean(aps)), pd.DataFrame(rows)

history = []
best_val_auc = -np.inf
start_epoch = 1
best_path = OUTPUT_DIR / 'best_densenet121_category_attention.pt'
last_path = OUTPUT_DIR / 'last_checkpoint.pt'
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

if RESUME_TRAINING:
    if not last_path.exists():
        raise FileNotFoundError(f'No epoch checkpoint to resume from: {last_path}')
    saved = torch.load(last_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(saved['model_state_dict'])
    optimizer.load_state_dict(saved['optimizer_state_dict'])
    scheduler.load_state_dict(saved['scheduler_state_dict'])
    scaler.load_state_dict(saved['scaler_state_dict'])
    history = saved['history']
    best_val_auc = saved['best_val_auroc']
    start_epoch = saved['epoch'] + 1
    print(f'Resuming after completed epoch {saved["epoch"]}; best validation AUROC={best_val_auc:.4f}')

for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss, seen = 0.0, 0
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}')
    for images, targets in progress:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda')):
            logits, _ = model(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * len(images)
        seen += len(images)
        progress.set_postfix(loss=f'{running_loss / max(seen, 1):.4f}')

    train_loss = running_loss / max(seen, 1)
    val_loss, y_val, p_val = collect_predictions(val_loader)
    val_auc, val_ap, _ = multilabel_metrics(y_val, p_val)
    scheduler.step(val_loss)
    row = {'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
           'val_macro_auroc': val_auc, 'val_macro_average_precision': val_ap,
           'learning_rate': optimizer.param_groups[0]['lr']}
    history.append(row)
    print(row)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'labels': LABELS,
            'image_size': IMAGE_SIZE,
            'use_pos_weight': USE_POS_WEIGHT,
            'validation_macro_auroc': val_auc,
            'seed': SEED,
        }, best_path)

    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'history': history,
        'best_val_auroc': best_val_auc,
        'labels': LABELS,
        'run_config': RUN_CONFIG,
    }, last_path)
    pd.DataFrame(history).to_csv(OUTPUT_DIR / 'epoch_history.csv', index=False)

history_df = pd.DataFrame(history)
display(history_df)
print('Best validation macro AUROC:', round(best_val_auc, 4))
print('Best checkpoint:', best_path)
print('Resume checkpoint:', last_path)
print('Run artifacts folder:', OUTPUT_DIR)


## 9. Evaluate the selected model once on the official held-out test patients

AUROC and average precision are reported by finding. These ranking metrics avoid a single arbitrary decision threshold, which is especially useful with uneven label frequencies. A missing class in a particular split would have an undefined metric (`NaN`); inspect positive counts alongside scores.

In [ ]:
checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
if checkpoint['labels'] != LABELS:
    raise ValueError('Checkpoint labels do not match this notebook')
model.load_state_dict(checkpoint['model_state_dict'])
test_loss, y_test, p_test = collect_predictions(test_loader)
test_macro_auc, test_macro_ap, test_results = multilabel_metrics(y_test, p_test)
print(f"Selected epoch: {checkpoint['epoch']}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test macro AUROC: {test_macro_auc:.4f}")
print(f"Test macro average precision: {test_macro_ap:.4f}")
display(test_results.sort_values('AUROC', ascending=False).reset_index(drop=True))
test_results.to_csv(OUTPUT_DIR / 'per_label_test_metrics.csv', index=False)
np.savez_compressed(OUTPUT_DIR / 'test_predictions.npz', y_true=y_test, y_score=p_test, labels=np.array(LABELS))


In [ ]:
# Per-finding ROC curves (only labels with both positive and negative examples are defined).
plt.figure(figsize=(9, 8))
for index, label in enumerate(LABELS):
    if np.unique(y_test[:, index]).size < 2:
        continue
    fpr, tpr, _ = roc_curve(y_test[:, index], p_test[:, index])
    auc = roc_auc_score(y_test[:, index], p_test[:, index])
    plt.plot(fpr, tpr, label=f'{label} ({auc:.3f})', linewidth=1.2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ChestX-ray14 test ROC curves')
plt.legend(loc='lower right', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()


## 10. Visualize one class-specific attention map

This overlays the learned spatial attention weights for one finding. The attention head forces each map to sum to one, but that does **not** prove it found the disease or provide a clinically validated localization. Compare it with radiologist-drawn boxes only if you have appropriate annotations and an evaluation protocol.

In [ ]:
# Choose an image from the official test split. Change this index if desired.
DISPLAY_INDEX = 0
DISPLAY_LABEL = 'Pneumonia'
row = test_df.iloc[DISPLAY_INDEX]
dataset_item = ChestXray14Dataset(test_df.iloc[[DISPLAY_INDEX]], training=False)
image_tensor, target_tensor = dataset_item[0]
label_index = LABELS.index(DISPLAY_LABEL)

model.eval()
with torch.no_grad():
    logits, attention_maps = model(image_tensor.unsqueeze(0).to(DEVICE))
    probabilities = torch.sigmoid(logits)[0].cpu().numpy()
    attention = attention_maps[0, label_index].unsqueeze(0).unsqueeze(0)
    attention = F.interpolate(attention, size=(IMAGE_SIZE, IMAGE_SIZE), mode='bilinear', align_corners=False)[0, 0].cpu().numpy()

image = image_tensor.permute(1, 2, 0).cpu().numpy() * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
image = np.clip(image, 0, 1)
attention = (attention - attention.min()) / (attention.max() - attention.min() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title(f"Image | {DISPLAY_LABEL} target={int(target_tensor[label_index])}")
axes[1].imshow(image, cmap='gray')
axes[1].imshow(attention, cmap='jet', alpha=0.45)
axes[1].set_title(f"Attention | model score={probabilities[label_index]:.3f}")
for axis in axes:
    axis.axis('off')
plt.tight_layout()
plt.show()


## 11. Save and cite

The run folder is printed near the top and contains `run_config.json`, `dataset_info.json`, split and label summaries, `last_checkpoint.pt`, `best_densenet121_category_attention.pt`, `epoch_history.csv`, `per_label_test_metrics.csv`, and `test_predictions.npz`. These files are created as the notebook runs, so a disconnect does not erase completed epochs when the output folder is on Drive.

### Saving the notebook and continuing later

- **Google Colab:** first choose **File → Save a copy in Drive**. The notebook code and displayed cell outputs are saved in that copy; model and metrics files go to `MyDrive/ChestXray14Runs/<RUN_NAME>/`. If the runtime stops, reopen the saved copy, keep the same `RUN_NAME`, set `RESUME_TRAINING=True`, and run cells from the top.
- **Kaggle Notebook:** add the dataset with **Add Input**, use `DATA_SOURCE='kaggle_input'`, then use **Save Version** after running. Files written under `/kaggle/working/<RUN_NAME>/` are notebook outputs; download them from the notebook Output panel or save a version to preserve them.
- **Local Jupyter:** save the `.ipynb` with Ctrl/Cmd+S. Results are written under `chestxray14_densenet_attention_seed42_v1/` in the working directory unless you change `RUN_NAME` or the output path.

After changing model settings, use a new `RUN_NAME` for a clean experiment. To resume, keep the same settings and name, set `RESUME_TRAINING=True`, and increase `EPOCHS` only if you want additional epochs. The notebook file and the saved model/results are separate artifacts; save both if you want the code, displayed outputs, and trained model together.

### Reading list

- Wang, X. et al. (2017). [ChestX-ray8: Hospital-scale Chest X-Ray Database and Benchmarks on Weakly-Supervised Classification and Localization of Common Thorax Diseases](https://openaccess.thecvf.com/content_cvpr_2017/html/Wang_ChestX-ray8_Hospital-Scale_Chest_CVPR_2017_paper.html). CVPR.
- Yao, L. et al. (2017). [Learning to diagnose from scratch by exploiting dependencies among labels](https://arxiv.org/abs/1710.10501).
- Rajpurkar, P. et al. (2017). [CheXNet: Radiologist-Level Pneumonia Detection on Chest X-Rays with Deep Learning](https://arxiv.org/abs/1711.05225).
- Ge, Z. et al. (2018). [Chest X-rays Classification: A Multi-Label and Fine-Grained Problem](https://arxiv.org/abs/1807.07247).
- Guan, Q. & Huang, Y. (2020). [Multi-label chest X-ray image classification via category-wise residual attention learning](https://doi.org/10.1016/j.patrec.2018.10.027). Pattern Recognition Letters.
- Wang, H. et al. (2021). [Triple attention learning for classification of 14 thoracic diseases using chest radiography](https://doi.org/10.1016/j.media.2020.101846). Medical Image Analysis.

### Interpretation limits

- This is a research-learning exercise, not a medical device or diagnostic tool.
- ChestX-ray14's report-mined labels have known noise and do not provide complete ground truth for every finding.
- AUROC values from this run are meaningful only with the exact split, preprocessing, labels, and model recorded; do not compare directly with papers unless their protocol is matched.
- Attention visualization is a model inspection aid, not evidence that a highlighted region contains pathology.
- Do not tune thresholds, settings, or epochs based on the final test set.